# GolStats - Bronze Events

This notebook is responsible for processing the raw event files
ingested from StatsBomb Open Data.

The Bronze layer stores the raw event data in Delta format while
adding basic metadata required for data lineage and traceability.

The main objectives are:

- Read the raw event JSON files.
- Identify the match associated with each event.
- Preserve the original event information.
- Store the resulting dataset as a Delta table.


## Bronze Layer - Event ingestion

Flow:

Raw JSON files
      ↓
Spark DataFrame
      ↓
Data lineage
(match_id + source_file)
      ↓
Delta Table
      ↓
golstats.bronze.eventos_statsbomb

## 1. Configuration

We define the location of the raw event files and the destination
table for the Bronze layer.

Keeping these values in variables makes the notebook easier to
maintain and reuse.

In [0]:
# Location of the raw StatsBomb event files.
VOLUME_PATH = "/Volumes/golstats/bronze/raw_files"

# Destination Bronze table.
BRONZE_TABLE = "golstats.bronze.eventos_statsbomb"

## 2. Read raw event files

The raw event data is stored as one JSON file per match.

Spark can read all event files as a single dataset using a file pattern.

At this stage, we do not apply business transformations. The goal is
to load the raw event structures into Spark so they can be stored in
the Bronze layer.

In [0]:
# Read all raw event JSON files from the Volume.
# The multiLine option is required because each StatsBomb JSON file
# contains the event data as a JSON array.

df_events = (
    spark.read
    .option("multiLine", True)
    .json(f"{VOLUME_PATH}/eventos_*.json")
)

display(df_events.head())


Row(50_50=None, bad_behaviour=None, ball_receipt=None, ball_recovery=None, block=None, carry=None, clearance=None, counterpress=None, dribble=None, duel=None, duration=0.0, foul_committed=None, foul_won=None, goalkeeper=None, id='3d695d99-d308-418c-bb6d-9b2a58607399', index=1, injury_stoppage=None, interception=None, location=None, minute=0, miscontrol=None, off_camera=None, out=None, pass=None, period=1, play_pattern=Row(id=1, name='Regular Play'), player=None, position=None, possession=1, possession_team=Row(id=771, name='France'), related_events=None, second=0, shot=None, substitution=None, tactics=Row(formation=4231, lineup=[Row(jersey_number=1, player=Row(id=3099, name='Hugo Lloris'), position=Row(id=1, name='Goalkeeper')), Row(jersey_number=2, player=Row(id=5476, name='Benjamin Pavard'), position=Row(id=2, name='Right Back')), Row(jersey_number=24, player=Row(id=11135, name='Ibrahima Konaté'), position=Row(id=3, name='Right Center Back')), Row(jersey_number=18, player=Row(id=8519

## 3. Validate the raw event data

Before writing the Bronze table, we perform basic validation on the
dataset.

The purpose of this step is to confirm that Spark successfully loaded
the event data and that the expected event structure is available.

In [0]:
# Count the total number of events loaded from the raw JSON files.

total_events = df_events.count()

print(f"Total events loaded: {total_events}")

Total events loaded: 35142


In [0]:
# Analyze the distribution of event types.

event_types = (
    df_events
    .groupBy("type.name")
    .count()
    .orderBy("count", ascending=False)
)

display(event_types)

name,count
Pass,10277
Ball Receipt*,9514
Carry,8171
Pressure,2332
Ball Recovery,892
Duel,640
Clearance,408
Block,380
Miscontrol,277
Foul Committed,272


## 4. Add data lineage information

The raw event files are stored separately for each match.

Before creating the Bronze table, we add metadata that identifies the
source file and the match associated with each event.

The match ID is extracted directly from the source filename.

This allows every event to be traced back to its original source.

In [0]:
from pyspark.sql.functions import col, regexp_extract

In [0]:
# Add the source file path for each event.
# Unity Catalog provides file metadata through the _metadata column.

df_events_with_lineage = (
    df_events
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )
)

In [0]:
# Extract the match ID from the source file path.
#
# Example:
# /Volumes/golstats/bronze/raw_files/eventos_3857286.json
#
# becomes:
# 3857286

df_events_with_lineage = (
    df_events_with_lineage
    .withColumn(
        "match_id",
        regexp_extract(
            col("source_file"),
            r"eventos_(\d+)\.json",
            1
        ).cast("long")
    )
)

In [0]:
display(
    df_events_with_lineage.select(
        "match_id",
        "source_file",
        "id",
        "minute",
        "team",
        "type"
    )
)

match_id,source_file,id,minute,team,type
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,3d695d99-d308-418c-bb6d-9b2a58607399,0,"List(771, France)","List(35, Starting XI)"
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,22342879-4d1b-49c0-af7a-fa11cd9cad82,0,"List(792, Australia)","List(35, Starting XI)"
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,667963b1-804c-4a31-9983-04109a2da3b8,0,"List(771, France)","List(18, Half Start)"
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,5a76e1c7-9941-4f1e-9ec1-e2d7aa80636e,0,"List(792, Australia)","List(18, Half Start)"
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,bfac39b6-016f-47cb-bb57-d04b4b6092b1,0,"List(771, France)","List(30, Pass)"
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,8e61023f-ff93-4f62-b163-33c01ec68f93,0,"List(771, France)","List(42, Ball Receipt*)"
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,de980b1a-a6d5-420d-9186-3897d2fa796a,0,"List(771, France)","List(43, Carry)"
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,204c9f66-a76d-49d1-96c9-6bc6eb43c3eb,0,"List(771, France)","List(30, Pass)"
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,3019ce25-b023-48f6-bf51-4ffcdf2a3951,0,"List(771, France)","List(42, Ball Receipt*)"
3857279,dbfs:/Volumes/golstats/bronze/raw_files/eventos_3857279.json,86dd3815-1c27-44d8-9970-755ed0575d8d,0,"List(771, France)","List(43, Carry)"


In [0]:
# Count the number of distinct matches loaded.

matches_loaded = (
    df_events_with_lineage
    .select("match_id")
    .distinct()
    .count()
)

print(f"Matches loaded into Bronze: {matches_loaded}")

Matches loaded into Bronze: 10


In [0]:
# Validate the number of events associated with each match.

display(
    df_events_with_lineage
    .groupBy("match_id")
    .count()
    .orderBy("match_id")
)

match_id,count
3857254,3680
3857265,3184
3857268,3499
3857271,3649
3857277,3719
3857279,3963
3857282,3687
3857285,3133
3857286,3299
3857300,3329


## 5. Create the Bronze Delta table

After validating the raw event data and adding data lineage information,
the dataset is stored as a Delta table in the Bronze layer.

The Bronze layer preserves the original event structure while providing
a reliable and queryable storage format for downstream transformations.

In [0]:
%sql

DROP TABLE IF EXISTS golstats.bronze.eventos_statsbomb;


In [0]:
# Write the validated raw events to the Bronze Delta table.
#
# The Bronze layer preserves the original StatsBomb structure.
# Only ingestion metadata was added:
# - match_id
# - source_file

(
    df_events_with_lineage
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("golstats.bronze.eventos_statsbomb")
)

In [0]:
%sql

SELECT 
    COUNT(*) AS total_events,
    COUNT(DISTINCT match_id) AS total_matches
FROM golstats.bronze.eventos_statsbomb;

total_events,total_matches
35142,10


Por lo tanto, el proyecto ahora está así:

![image_1788520728338.png](./image_1788520728338.png "image_1788520728338.png")